# 01 - Análisis exploratorio de datos
**Proyecto de aula - Modelos y Simulación de Sistemas II**

Integrantes: [nombres]

En este notebook revisamos cómo es la base de datos de fraude con tarjetas de crédito antes de entrenar modelos.
Los datos se descargan de Kaggle: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
y el archivo `creditcard.csv` debe quedar en la carpeta `data/`.

## 0. Antes de correr este notebook

Este notebook busca el archivo de datos automáticamente en varios lugares
comunes (carpeta local `data/`, Google Drive en Colab). Si no lo encuentra,
te va a pedir que lo subas.

No hace falta que edites nada: solo corre las celdas en orden, de arriba
hacia abajo.

In [ ]:
import os

RUTA_DATOS = None
candidatos = [
    "../data/creditcard.csv",
    "data/creditcard.csv",
    "/content/drive/MyDrive/creditcard.csv",
    "/content/drive/MyDrive/data/creditcard.csv",
    "/content/drive/MyDrive/Colab Notebooks/data/creditcard.csv",
    "creditcard.csv",
]

for ruta in candidatos:
    if os.path.exists(ruta):
        RUTA_DATOS = ruta
        break

# Si no se encontró y estamos en Google Colab, ofrecer montar Drive o subir el archivo
if RUTA_DATOS is None:
    try:
        import google.colab
        en_colab = True
    except ImportError:
        en_colab = False

    if en_colab:
        from google.colab import drive
        print("No se encontró el archivo. Intentando conectar Google Drive...")
        drive.mount('/content/drive')
        for ruta in candidatos:
            if os.path.exists(ruta):
                RUTA_DATOS = ruta
                break
        if RUTA_DATOS is None:
            print("No se encontró en Drive. Sube el archivo creditcard.csv manualmente:")
            from google.colab import files
            subido = files.upload()
            RUTA_DATOS = list(subido.keys())[0]

if RUTA_DATOS is None:
    raise FileNotFoundError(
        "No se encontró creditcard.csv. Revisa el README de la carpeta data/ "
        "para saber cómo descargarlo y dónde colocarlo."
    )

print("Usando el archivo de datos en:", RUTA_DATOS)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_style("whitegrid")
CARPETA_RESULTADOS = "../resultados" if os.path.isdir("..") and os.path.isdir("../notebooks") else "resultados"
try:
    os.makedirs(CARPETA_RESULTADOS, exist_ok=True)
except Exception:
    CARPETA_RESULTADOS = "."

## 1. Cargar los datos

In [ ]:
df = pd.read_csv(RUTA_DATOS)
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])
df.head()

## 2. Qué variables hay
Son 30 variables de entrada y la etiqueta `Class` (0 = normal, 1 = fraude).
- `Time`: segundos desde la primera transacción
- `V1` a `V28`: variables anónimas (ya vienen transformadas con PCA por confidencialidad)
- `Amount`: monto de la transacción

In [ ]:
df.info()

In [ ]:
df.describe().T

## 3. Datos faltantes y duplicados

In [ ]:
print("Datos faltantes en total:", df.isna().sum().sum())
print("Filas duplicadas:", df.duplicated().sum())

Anotar aquí lo que salió y qué decidimos hacer con los duplicados (dejarlos o quitarlos) y por qué:

- ...

## 4. Balance de clases

In [ ]:
conteo = df["Class"].value_counts()
porcentaje = df["Class"].value_counts(normalize=True) * 100
print(conteo)
print()
print(porcentaje.round(3))

plt.figure(figsize=(5, 4))
ax = sns.countplot(x="Class", data=df)
ax.set_yscale("log")
plt.xticks([0, 1], ["Normal (0)", "Fraude (1)"])
plt.xlabel("Clase")
plt.ylabel("Número de transacciones (escala log)")
plt.title("Distribución de clases")
plt.tight_layout()
plt.savefig(f"{CARPETA_RESULTADOS}/distribucion_clases.png", dpi=200)
plt.show()

Solo una parte muy pequeña de las transacciones son fraude, así que la exactitud (accuracy) no sirve como medida principal: un modelo que diga siempre "normal" acertaría casi el 100 %. Por eso más adelante usamos precisión, recall, F1 y AUPRC.

## 5. Monto de las transacciones (`Amount`)

In [ ]:
print(df.groupby("Class")["Amount"].describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for clase, nombre, ax in [(0, "Normal", axes[0]), (1, "Fraude", axes[1])]:
    ax.hist(df.loc[df["Class"] == clase, "Amount"], bins=50)
    ax.set_title(f"Monto - {nombre}")
    ax.set_xlabel("Monto")
    ax.set_ylabel("Número de transacciones")
    ax.set_yscale("log")
plt.tight_layout()
plt.savefig(f"{CARPETA_RESULTADOS}/monto_por_clase.png", dpi=200)
plt.show()

## 6. Tiempo (`Time`)

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df.loc[df["Class"] == 0, "Time"] / 3600, bins=48, alpha=0.6, label="Normal", density=True)
plt.hist(df.loc[df["Class"] == 1, "Time"] / 3600, bins=48, alpha=0.6, label="Fraude", density=True)
plt.xlabel("Horas desde la primera transacción")
plt.ylabel("Densidad")
plt.title("Transacciones a lo largo del tiempo")
plt.legend()
plt.tight_layout()
plt.savefig(f"{CARPETA_RESULTADOS}/tiempo_por_clase.png", dpi=200)
plt.show()

## 7. Relación de cada variable con la clase

In [ ]:
corr = df.corr()["Class"].drop("Class").sort_values()

plt.figure(figsize=(6, 8))
corr.plot(kind="barh")
plt.xlabel("Correlación con Class")
plt.ylabel("Variable")
plt.title("Correlación de cada variable con el fraude")
plt.tight_layout()
plt.savefig(f"{CARPETA_RESULTADOS}/correlacion_con_class.png", dpi=200)
plt.show()

print("Más relacionadas con fraude (en valor absoluto):")
print(corr.abs().sort_values(ascending=False).head(10).round(3))

## 8. Conclusiones del análisis
Completar con lo que observamos:

- Tamaño de la base y número de fraudes:
- Datos faltantes / duplicados:
- Variables que más se relacionan con el fraude:
- Cómo se ve el monto de los fraudes frente a los normales:
- Qué implica el desbalance para el resto del proyecto: